# Import

In [0]:
import requests
import json
import re

#Configure

In [0]:
base_url = "https://honolulu-api.datausa.io/tesseract/data.jsonrecords?cube=acs_yg_total_population_1&drilldowns=Year%2CNation&locale=en&measures=Population"
all_records = []
offset = 0
limit = 50

# Process

##1a. get

In [0]:
response = requests.get(base_url, timeout=10)
response.raise_for_status()
result_of_api_call = response.json()

##1b. get (with pagination)

In [0]:
while True:
    # Add pagination parameters
    url = f"{base_url}&limit={limit}&offset={offset}"
    
    print(f"Fetching records {offset} to {offset + limit}...")
    response = requests.get(url, timeout=10)
    
    # Check if request was successful
    response.raise_for_status()
    
    # Parse JSON response
    data = response.json()
    
    # Extract records and pagination info
    records = data.get('data', [])
    page_info = data.get('page', {})
    total_records = page_info.get('total', 0)
    
    print(f"  Retrieved {len(records)} records")
    
    # Add records to our collection
    all_records.extend(records)
    
    # Check if we've got all records
    if offset + len(records) >= total_records or len(records) == 0:
        break
    
    # Move to next page
    offset += limit

In [0]:
all_records

##2a. write

In [0]:
dbutils.fs.put('s3://rearc-quest-107628756615-us-east-2-an/datausa/annual_us_pop_2013_thru_2024.json', json.dumps(result_of_api_call), overwrite=True)

## 2b. write (delta)

In [0]:
def clean_column_name(col_name):
    """
    Remove or replace invalid characters from a column name.
    Databricks Delta doesn't allow: space , ; { } ( ) \n \t =
    
    Args:
        col_name: Original column name
        
    Returns:
        Cleaned column name
    """
    # Replace invalid characters with underscores
    invalid_chars = r'[ ,;{}()\n\t=]'
    cleaned = re.sub(invalid_chars, '_', col_name)
    
    # Remove consecutive underscores
    cleaned = re.sub(r'_+', '_', cleaned)
    
    # Remove leading/trailing underscores
    cleaned = cleaned.strip('_')
    
    # If empty after cleaning, use 'col_N' as fallback
    if not cleaned:
        cleaned = 'col_unnamed'
    
    return cleaned

def clean_dataframe_schema_dict(df):
    """
    Alternative approach using withColumnsRenamed (PySpark 3.4+).
    More efficient for many columns.
    
    Args:
        df: PySpark DataFrame with potentially invalid column names
        
    Returns:
        DataFrame with cleaned column names
    """
    rename_map = {
        old_col: clean_column_name(old_col) 
        for old_col in df.columns
    }
    
    # Filter to only columns that actually need renaming
    rename_map = {k: v for k, v in rename_map.items() if k != v}
    
    if rename_map:
        return df.withColumnsRenamed(rename_map)
    return df

In [0]:
all_records_df = spark.createDataFrame(all_records)

cleaned_df = clean_dataframe_schema_dict(all_records_df)

cleaned_df.write.mode("overwrite").saveAsTable("quest.annual_us_pop_2013_thru_2024")